# Demo 01 — Zero-Shot Prompting with Groq

**Option 1: Use the Groq API (recommended for this demo).**

1. Install the official `groq` Python library.
2. Set your `GROQ_API_KEY` using a system environment variable, Colab Secrets, or a hidden prompt.
3. Define a reusable `get_completion()` function.
4. Run the zero-shot sentiment-classification prompt.

**Google Colab:** Open the 🔑 Secrets panel, add a secret named `GROQ_API_KEY`, and grant this notebook access. **VS Code:** set the `GROQ_API_KEY` environment variable before starting the notebook kernel. If neither is available, the setup cell securely prompts for the key (it will not print the key). Do **not** hardcode or share API keys.

> A Groq API key is required to execute Option 1. API calls may incur usage charges or rate limits.


In [5]:
# Step 1: Install and configure the Groq API client
%pip install -q groq
import os
from dotenv import load_dotenv
from groq import Groq

# Step 1: Load variables from .env file
load_dotenv()

# Step 2: Read the Groq API key
api_key = os.getenv("GROQ_API_KEY")

# Step 3: Validate the key
if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env file")

# Step 4: Initialize Groq client
client = Groq(api_key=api_key)

MODEL_NAME = "openai/gpt-oss-20b"

print(f"Groq client ready. Model: {MODEL_NAME}")


[notice] A new release of pip is available: 26.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Groq client ready. Model: openai/gpt-oss-20b


In [6]:
def get_completion(prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content


# Zero-shot sentiment classification
prompt = """
Classify the sentiment of the following review
as Positive, Negative, or Neutral.

Review: The product quality is excellent
and delivery was very fast.

Return only the sentiment.
"""

result = get_completion(prompt)

print("Sentiment:", result)

Sentiment: Positive


## Option 2 — Run a Hugging Face model locally in the notebook runtime (no Groq API)

This alternative is **independent of Groq** and is preserved from the original notebook for comparing API-hosted versus locally executed models.

1. Install `transformers`, `torch`, and `accelerate`.
2. Download the example Hugging Face model to the runtime.
3. Run the same kind of zero-shot sentiment-classification task.

In Google Colab, this runs on the Colab runtime CPU/GPU, not through Groq. Downloads need internet access and model execution consumes runtime resources.


In [7]:
!pip install -q transformers torch accelerate

import torch
from transformers import pipeline

# Runs directly in Colab memory/GPU, no API endpoints needed
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    #model="meta-llama/Llama-3.2-1B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

messages = [
    {
        "role": "user",
        "content": (
            "Classify the sentiment of the following customer review into one word: "
            "Positive, Neutral, or Negative.\n\n"
            "Review: 'The item arrived early, but the instructions were missing.'\n"
            "Sentiment:"
        )
    }
]

output = generator(messages, max_new_tokens=15, do_sample=False)
print(output[0]["generated_text"][-1]["content"].strip())

#note that- the result might be vary depending upon the models that are using


[notice] A new release of pip is available: 26.0 -> 26.3
[notice] To update, run: pip install --upgrade pip


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Neutral
